In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")

catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "220_build_silver_closed_bids_vendor_item_latest.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.silver.closed_bids_base_clean"
TARGET_TABLE = f"{catalog}.silver.closed_bids_vendor_item_latest"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build silver.closed_bids_vendor_item_latest
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    USING DELTA
    AS
    WITH vendor_rows AS (
        SELECT
              *
            , CASE
                  WHEN specification_description IS NOT NULL
                    THEN specification_description
                  WHEN COALESCE(bid_code, '') LIKE '9602-%'
                    THEN bid_item_description
                  WHEN COALESCE(bid_code, '') LIKE '9606-%'
                    THEN bid_item_description
                  WHEN UPPER(COALESCE(bid_item_description, '')) LIKE '%PAYMENT ADJUSTMENT%'
                    THEN bid_item_description
                  WHEN UPPER(COALESCE(bid_item_description, '')) LIKE '%SPECIAL DEDUCTION%'
                    THEN bid_item_description
                  ELSE NULL
              END AS effective_specification_description
        FROM {SOURCE_TABLE}
        WHERE row_type = 'VENDOR'
          AND is_engineer_estimate_row = FALSE
          AND project_id IS NOT NULL
          AND vendor_name IS NOT NULL
          AND bid_item_sequence_number IS NOT NULL
    )

    , vendor_rows_keyed AS (
        SELECT
              *
            , CASE
                  WHEN specification_code IS NOT NULL
                   AND effective_specification_description IS NOT NULL
                   AND measurement_unit IS NOT NULL
                  THEN md5(
                      concat_ws(
                            '|'
                          , COALESCE(CAST(specification_code AS STRING), '')
                          , COALESCE(TRIM(UPPER(effective_specification_description)), '')
                          , COALESCE(TRIM(UPPER(measurement_unit)), '')
                      )
                    )
                  ELSE NULL
              END AS item_specification_key
        FROM vendor_rows
    )

    , vendor_project_stats AS (
        SELECT
              project_id
            , vendor_name
            , COUNT(DISTINCT bid_submit_sequence_number) AS vendor_project_submit_sequence_count
            , MAX(bid_submit_sequence_number) AS latest_submit_for_vendor_project
        FROM vendor_rows_keyed
        GROUP BY
              project_id
            , vendor_name
    )

    , item_history AS (
        SELECT
              v.*
            , ROW_NUMBER() OVER (
                PARTITION BY v.project_id, v.vendor_name, v.bid_item_sequence_number
                ORDER BY
                      v.bid_submit_sequence_number DESC NULLS LAST
                    , v.source_updated_at DESC NULLS LAST
                    , v.ingested_at_utc DESC NULLS LAST
                    , v.source_row_id DESC NULLS LAST
              ) AS item_latest_rank
            , COUNT(*) OVER (
                PARTITION BY v.project_id, v.vendor_name, v.bid_item_sequence_number
              ) AS item_version_count
            , MIN(v.bid_submit_sequence_number) OVER (
                PARTITION BY v.project_id, v.vendor_name, v.bid_item_sequence_number
              ) AS min_submit_sequence_for_item
            , MAX(v.bid_submit_sequence_number) OVER (
                PARTITION BY v.project_id, v.vendor_name, v.bid_item_sequence_number
              ) AS max_submit_sequence_for_item
            , MIN(v.bid_item_quantity) OVER (
                PARTITION BY v.project_id, v.vendor_name, v.bid_item_sequence_number
              ) AS min_item_quantity_for_item
            , MAX(v.bid_item_quantity) OVER (
                PARTITION BY v.project_id, v.vendor_name, v.bid_item_sequence_number
              ) AS max_item_quantity_for_item
            , MIN(v.bid_item_unit_price_amount) OVER (
                PARTITION BY v.project_id, v.vendor_name, v.bid_item_sequence_number
              ) AS min_item_unit_price_for_item
            , MAX(v.bid_item_unit_price_amount) OVER (
                PARTITION BY v.project_id, v.vendor_name, v.bid_item_sequence_number
              ) AS max_item_unit_price_for_item
        FROM vendor_rows_keyed v
    )

    SELECT
          h.source_name
        , h.base_row_key
        , h.source_row_id
        , h.source_created_at
        , h.source_updated_at
        , h.source_version
        , h.ingestion_run_id
        , h.ingested_at_utc
        , h.project_id
        , h.project_name
        , h.project_number
        , h.state_project_number
        , h.control_section_job_csj
        , h.controlling_project_id_ccsj
        , h.project_type
        , h.project_classification
        , h.project_actual_let_date
        , h.project_estimated_let_date
        , h.project_limits_from
        , h.project_limits_to
        , h.county
        , h.district_division
        , h.highway
        , h.short_description
        , h.federal_project_number
        , h.vendor_name
        , h.vendor_name_standardized
        , h.is_engineer_estimate_row
        , h.bid_submit_sequence_number
        , h.bid_rank_sequence_number
        , h.low_bidder_flag
        , h.dbe_goal_percent
        , h.maximum_number_of_working
        , h.bid_code
        , h.alternative_bid_code
        , h.bid_item_sequence_number
        , h.bid_item_description
        , h.measurement_unit
        , h.bid_item_quantity
        , h.bid_item_unit_price_amount
        , h.bid_total_amount
        , h.specification_code
        , h.specification_description
        , h.effective_specification_description
        , h.spec_book_year
        , h.item_specification_key
        , h.engineer_s_estimate_unit
        , h.sealed_engineer_s_estimate
        , h.sealed_engineer_s_estimate_1
        , h.a_b_engineer_s_estimate_amount
        , h.engineer_estimate_unit_price
        , h.engineer_project_total
        , h.force_account_amount
        , h.force_account_description
        , h.nbi_number
        , h.utility_id
        , h.row_type
        , h.is_valid_quantity
        , h.is_valid_unit_price
        , h.is_valid_bid_total
        , h.business_submission_key
        , h.business_item_key
        , h.is_duplicate_business_item_key
        , h.record_hash
        , h.is_current_row
        , h.dedupe_rank
        , h.item_latest_rank
        , h.item_version_count
        , h.min_submit_sequence_for_item
        , h.max_submit_sequence_for_item
        , s.vendor_project_submit_sequence_count
        , s.latest_submit_for_vendor_project
        , CASE WHEN s.vendor_project_submit_sequence_count > 1 THEN TRUE ELSE FALSE END AS vendor_project_has_resubmission_history
        , CASE WHEN h.item_version_count > 1 THEN TRUE ELSE FALSE END AS item_has_resubmission_history
        , CASE WHEN h.min_submit_sequence_for_item <> h.max_submit_sequence_for_item THEN TRUE ELSE FALSE END AS item_changed_across_submit_sequences
        , CASE WHEN h.bid_submit_sequence_number = s.latest_submit_for_vendor_project THEN TRUE ELSE FALSE END AS is_latest_submit_for_vendor_project
        , CASE WHEN h.max_submit_sequence_for_item < s.latest_submit_for_vendor_project THEN TRUE ELSE FALSE END AS item_missing_from_latest_project_submit
        , CASE
              WHEN COALESCE(h.min_item_quantity_for_item, CAST(-999999999.999 AS DECIMAL(38,10))) <> COALESCE(h.max_item_quantity_for_item, CAST(-999999999.999 AS DECIMAL(38,10)))
                OR COALESCE(h.min_item_unit_price_for_item, CAST(-999999999.999 AS DECIMAL(38,10))) <> COALESCE(h.max_item_unit_price_for_item, CAST(-999999999.999 AS DECIMAL(38,10)))
              THEN TRUE ELSE FALSE
          END AS item_changed_value_across_submits
    FROM item_history h
    INNER JOIN vendor_project_stats s
        ON h.project_id = s.project_id
       AND h.vendor_name = s.vendor_name
    WHERE h.item_latest_rank = 1
    """)

    spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS
    'Current vendor-item silver table built from silver.closed_bids_base_clean. Preserves the latest known state per project, vendor, and bid item sequence and includes resubmission diagnostics.'
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_TABLE}").collect()[0]["row_count"]

    log_run(
        "SUCCESS",
        row_count,
        f"Built {TARGET_TABLE} successfully."
    )

    print("=" * 70)
    print(f"Built {TARGET_TABLE}")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise